# A decision list you can walk by hand

The counts suggest five steps, in this order. The first step that matches is the prediction.

1. If `CryoSleep` is `True`, predict transported.
2. If `CryoSleep` is `False` and the luxury bill is known and at least 500, predict stayed.
3. If `Age` is known and at most 12, predict transported.
4. If `HomePlanet` is `Europa` and the deck is `B` or `C`, predict transported.
5. Otherwise predict stayed.

Step 2 does not change the cryosleep rule on its own: an awake passenger is already predicted to stay. It stops a later step from marking a high luxury bill as transported. No awake child has a luxury bill of 500 or more, so step 2 and step 3 do not compete over children.

Walk the named rows.

Candra, `0006_02`: cryosleep is `True`. Step 1 predicts transported. Her fate is transported.

Maham, `0001_01`: awake, luxury bill $0$, age 39, Europa, deck B. Steps 1 to 3 do not match. Step 4 predicts transported. Her fate is stayed. The list is wrong. The cryosleep rule, which stopped at "awake, so stayed," is right.

Juanna, `0002_01`: awake, luxury bill $702 \ge 500$. Step 2 predicts stayed. Her fate is transported. Both rules are wrong.

Altark, `0003_01`: luxury bill $6807 \ge 500$. Step 2 predicts stayed. His fate is stayed.

Solam, `0003_02`: luxury bill $3522 \ge 500$. Step 2 predicts stayed. His fate is stayed.

Nelly, `0013_01`: cryosleep is `True`. Step 1 predicts transported. There is no fate to compare. The sample file says stayed.

On all 8693 known fates the list is right on 6503 rows.

$$
6503 - 6244 = 259.
$$

Those 259 are the net gain over the cryosleep rule. The next lesson opens that net into correct flips and wrong flips.


In [2]:
# Locate the contest tables beside this notebook, or under the course root.
from pathlib import Path
import csv

def manifest_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "13-Spaceship Titanic" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "sample_submission.csv").exists():
            return direct
    raise FileNotFoundError(name)

def read_manifest(name):
    with manifest_path(name).open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))

train = read_manifest("train.csv")
test = read_manifest("test.csv")
submission = read_manifest("sample_submission.csv")
print(len(train), len(test), len(submission))
print(train[0]["PassengerId"], train[0]["Name"])


8693 4277 4277
0001_01 Maham Ofracculy


In [3]:
# Whole numbers are stored as decimals, such as 39.0.
def whole(text):
    return int(float(text))

def luxury(row):
    columns = ("RoomService", "Spa", "VRDeck")
    if any(row[column] == "" for column in columns):
        return None
    return sum(whole(row[column]) for column in columns)

def cryo_rule(row):
    return row["CryoSleep"] == "True"

def decision_list(row):
    # Step 1. Cryosleep.
    if row["CryoSleep"] == "True":
        return True
    # Step 2. Awake, and the luxury bill is at least 500.
    bill = luxury(row)
    if row["CryoSleep"] == "False" and bill is not None and bill >= 500:
        return False
    # Step 3. Age at most 12.
    if row["Age"] != "" and whole(row["Age"]) <= 12:
        return True
    # Step 4. Europa, deck B or deck C.
    if row["Cabin"] != "":
        deck = row["Cabin"].split("/")[0]
        if row["HomePlanet"] == "Europa" and deck in ("B", "C"):
            return True
    # Step 5. Everyone else stays.
    return False

print("rules ready", len(train))


rules ready 8693


In [4]:
# Walk the named rows, then score the whole training manifest.
by_id = {row["PassengerId"]: row for row in train}
cases = [
    ("0001_01", True, False),
    ("0002_01", False, True),
    ("0003_01", False, False),
    ("0003_02", False, False),
    ("0006_02", True, True),
]
for passenger_id, prediction, fate in cases:
    row = by_id[passenger_id]
    print(row["Name"], decision_list(row), row["Transported"])
    assert decision_list(row) is prediction
    assert (row["Transported"] == "True") is fate
print("Nelly", decision_list(test[0]), submission[0]["Transported"])
assert decision_list(test[0]) is True
assert submission[0]["Transported"] == "False"

awake_children = [
    row
    for row in train
    if row["Age"] != ""
    and whole(row["Age"]) <= 12
    and row["CryoSleep"] == "False"
]
high_child_bills = sum(
    luxury(row) is not None and luxury(row) >= 500 for row in awake_children
)
print("awake children", len(awake_children), "high luxury", high_child_bills)
assert high_child_bills == 0

correct = sum(
    decision_list(row) == (row["Transported"] == "True") for row in train
)
cryo_correct = sum(
    cryo_rule(row) == (row["Transported"] == "True") for row in train
)
print(correct, cryo_correct, correct - cryo_correct)
assert cryo_correct == 6244
assert correct == 6503
assert correct - cryo_correct == 259


Maham Ofracculy True False
Juanna Vines False True
Altark Susent False False
Solam Susent False False
Candra Jacostaffey True True
Nelly True False
awake children 406 high luxury 0
6503 6244 259


## Pitfall

The list scores 6503 and still misses Maham, the first passenger in the file. Step 4 reads "Europa, deck B" as transported. Her row is the counterexample. A higher total does not make every earlier rule worse on every person: the cryosleep rule is right on Maham, and the list is wrong.

## Summary

Five steps, first match wins, 6503 of 8693. The net gain over cryosleep alone is 259. Nelly is predicted transported, and her fate is not in the test file.
